In [1]:
import gc
import os
import sys
import psutil
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() - 1 or 1
_TOTAL_MEMORY = psutil.virtual_memory().total
_AVAILABLE_MEMORY = psutil.virtual_memory().available
_MEMORY_PER_WORKER = max(100 * 1024**2, _AVAILABLE_MEMORY // (_NCPU + 1))
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | {_TOTAL_MEMORY / 1024**3:.1f}GB total RAM ({_AVAILABLE_MEMORY / 1024**3:.1f}GB available) | pyarrow {pa.__version__}", flush=True)


Running with 47 CPU cores | 92.3GB total RAM (46.5GB available) | pyarrow 24.0.0


Variables

In [2]:
%run ../0_Config/0_variables.ipynb

Aggregate targets to output resolution

In [3]:
import time as _time

def build_targets(
        state: str,
        horizon_hours: int,
        targets_path: Path,
        pipeline_start: pd.Timestamp,
        pipeline_end: pd.Timestamp,
) -> pd.DataFrame:
    # Calculate horizon range at 5 minute granularity
    horizons = horizon_hours * 12  #### 48 * (60 / 5) == 576 horizons

    _t = _time.perf_counter()
    print(f"Loading targets parquet: {targets_path}", flush=True)
    targets = pd.read_parquet(
        targets_path,
        filters=[
            ("SETTLEMENTDATE", ">=", pipeline_start),
            ("SETTLEMENTDATE", "<=", pipeline_end + pd.Timedelta(days=3)),
        ],
    )
    print(f"  loaded targets: shape={targets.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

    # Build shifted target matrix using numpy (float32) to avoid ~3 GiB float64 allocation
    price_vals = targets[state.lower() + "_price"].values.astype(np.float32)
    n = len(price_vals)
    print(f"  building shifted target matrix: {n:,} rows × {horizons} horizons (~{n * horizons * 4 / 1024**2:.0f}MB float32)", flush=True)
    _t = _time.perf_counter()
    shifted = np.empty((n, horizons), dtype=np.float32)
    for h in tqdm(range(horizons), desc="shifting horizons", leave=False):
        shifted[:n - h, h] = price_vals[h:]
        shifted[n - h:, h] = np.nan
    print(f"  shift complete in {_time.perf_counter() - _t:.1f}s", flush=True)

    future_prediction_targets = pd.DataFrame(
        shifted,
        index=targets.index,
        columns=[f"target_h{h + 1}" for h in range(horizons)],
    )
    del shifted, price_vals
    gc.collect()

    return future_prediction_targets.loc[:pipeline_end]


future_prediction_targets = build_targets(
    state=os.environ["TARGET"],
    horizon_hours=int(os.environ["HORIZON_HOURS"]),
    targets_path=os.environ["FEATURES_PATH"],
    pipeline_start=pd.Timestamp(os.environ["FEATURE_DATASET_START"]),
    pipeline_end=pd.Timestamp(os.environ["FEATIRE_DATASET_END"]),
)

def aggregate_targets(future_prediction_targets):
    OUTPUT_RESOLUTION = int(os.environ["OUTPUT_RESOLUTION"])

    target_cols = list(future_prediction_targets.columns)
    horizons = len(target_cols)
    bucket_size = max(1, OUTPUT_RESOLUTION // 5)
    n_buckets = horizons // bucket_size
    n_raw = n_buckets * bucket_size

    print(
        f"Aggregating targets: {horizons} raw horizons → {n_buckets} buckets "
        f"of {bucket_size} (output resolution {OUTPUT_RESOLUTION}min)",
        flush=True,
    )
    _t = _time.perf_counter()
    values = future_prediction_targets.values.astype(np.float32)
    aggregated = values[:, :n_raw].reshape(len(values), n_buckets, bucket_size).mean(axis=2)
    del values
    gc.collect()
    print(f"  aggregation done in {_time.perf_counter() - _t:.1f}s", flush=True)
    bucket_cols = [f"h{i + 1}" for i in range(n_buckets)]

    future_prediction_targets_agg = pd.DataFrame(
        aggregated,
        index=future_prediction_targets.index,
        columns=bucket_cols,
    )
    display(future_prediction_targets_agg[:3])

    return future_prediction_targets_agg


future_prediction_targets_agg = aggregate_targets(future_prediction_targets)
del future_prediction_targets
gc.collect()

_stem = Path(os.environ['FEATURE_DATASET']).stem
_out_path = resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_targets_agg_{_stem}.parquet")
print(f"Writing aggregated targets parquet: {_out_path}", flush=True)
_t = _time.perf_counter()
future_prediction_targets_agg.to_parquet(_out_path)
print(f"  write done in {_time.perf_counter() - _t:.1f}s", flush=True)


Loading targets parquet: /home/ec2-user/Forecasting/3_Features_select/../2_Features_build/Feature_data/1_dispatch_price.parquet


  loaded targets: shape=(736417, 638) in 13.8s
  building shifted target matrix: 736,417 rows × 576 horizons (~1618MB float32)


  shift complete in 7.1s


Aggregating targets: 576 raw horizons → 96 buckets of 6 (output resolution 30min)
  aggregation done in 1.4s


,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,h25,h26,h27,h28,h29,h30,h31,h32,h33,h34,h35,h36,h37,h38,h39,h40,h41,h42,h43,h44,h45,h46,h47,h48,h49,h50,h51,h52,h53,h54,h55,h56,h57,h58,h59,h60,h61,h62,h63,h64,h65,h66,h67,h68,h69,h70,h71,h72,h73,h74,h75,h76,h77,h78,h79,h80,h81,h82,h83,h84,h85,h86,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00,66.771278,68.736610,71.561913,71.393814,67.557159,64.376251,62.394154,51.286774,53.846653,50.875000,47.969280,48.631649,53.432526,45.549961,48.958332,51.223370,51.833328,48.756680,60.809326,59.242542,65.243538,68.419991,75.266632,87.591408,91.870003,91.864998,96.619423,93.356079,97.833595,91.971046,92.450439,103.648804,96.037132,98.407753,94.583717,108.256554,131.045959,128.823776,115.016685,100.055275,103.012024,100.274956,92.320435,73.138535,99.104866,82.169136,73.394318,77.301506,75.259422,64.210213,59.465015,47.683834,51.239410,50.860371,59.252514,62.060055,64.291313,65.579994,71.565880,81.070984,91.964966,91.866676,95.826576,100.625389,93.834862,91.866402,93.166344,91.597313,102.530182,104.611305,99.189972,119.451965,107.280708,190.787033,116.456909,109.511574,119.361382,109.234642,95.479492,90.175728,95.548737,116.989037,106.786232,117.873390,113.989891,108.508858,92.866425,97.089691,103.439690,88.767456,92.471924,84.447395,83.185814,77.843376,82.700470,83.707741
2019-01-01 00:05:00,66.948692,69.325172,72.795120,70.308716,66.977119,63.867970,62.197407,50.223507,53.750000,50.291199,47.594746,48.631649,53.165852,45.816639,48.958332,52.181702,48.208328,53.693905,61.014042,59.082264,66.331871,68.419983,77.274132,89.492249,91.870003,91.865013,96.619408,95.499756,95.796013,92.960350,91.353424,107.507141,101.868797,90.780479,93.913124,116.015472,123.430626,131.148041,112.496300,98.715637,103.012024,98.463295,90.303764,71.576836,103.538185,77.847908,74.765587,77.301506,75.448257,62.531376,59.465015,45.770496,46.445858,56.444309,59.420460,63.416718,66.012993,65.709991,71.590874,84.189323,91.966637,93.946388,95.323128,100.596924,92.285393,91.866402,93.166344,91.739571,106.246269,102.106598,98.835594,122.312744,113.110687,186.801590,112.145149,110.525116,120.815010,108.244659,92.611168,90.630096,100.930061,115.010002,106.786232,123.650879,106.240868,107.143730,91.706970,100.394188,100.545265,88.016846,92.490990,82.436005,82.249817,77.553726,85.163582,81.877991
2019-01-01 00:10:00,67.518059,68.807365,73.763771,69.035126,66.897430,63.167255,59.882412,51.181843,53.749989,46.666153,50.261459,49.744980,51.004200,46.864960,48.691692,52.448345,48.208332,55.360569,60.305710,60.450977,67.408142,68.992020,79.430977,90.671715,91.868340,91.865013,96.619408,96.980896,94.316528,93.281982,91.520119,107.017143,100.057121,96.450478,91.328819,122.128685,125.733406,125.318054,111.361259,99.099724,101.881401,96.484871,89.468361,76.272255,98.524605,75.094818,78.341927,75.372368,72.712700,62.531399,59.063328,43.857162,46.445877,57.402622,61.157246,62.240479,66.160767,68.209999,70.827545,88.095978,91.968300,93.944733,95.323120,100.596924,92.285400,91.866394,93.166351,91.695747,107.887108,104.367897,96.085060,125.064980,118.940666,180.969925,109.530724,110.182228,122.188683,106.405327,91.435066,90.392509,105.766731,114.030113,107.766121,123.650879,106.458649,103.589272,91.037514,104.401970,97.206940,86.694427,92.930786,82.382881,81.375237,78.000168,85.375366,77.760040


Writing aggregated targets parquet: /home/ec2-user/Forecasting/3_Features_select/../3_Features_select/Selected_features/NSW_targets_agg_1_dispatch_price.parquet
  write done in 8.6s


In [ ]:
%reset -f